# Notebook guiado — K-Nearest Neighbors (KNN) desde cero

**RA1 · Programación de Inteligencia Artificial**  
**Notebook de refuerzo · Google Colab**

En este notebook aprenderemos KNN de forma visual y progresiva.

La idea principal del algoritmo es sencilla:

> **Para predecir una nueva observación, buscamos las observaciones conocidas más cercanas y utilizamos sus respuestas como referencia.**

## Recorrido

1. Qué es KNN
2. Qué significa ser “vecino”
3. Distancia euclídea
4. Predicción manual paso a paso
5. Qué representa `k`
6. Por qué el escalado es importante
7. KNN real con Iris
8. Train / test
9. Pipeline con `StandardScaler`
10. Evaluación y matriz de confusión
11. Elegir `k` con validación cruzada
12. Fronteras de decisión
13. Inspeccionar los vecinos
14. Pesos y métricas de distancia
15. Limitaciones
16. KNN para regresión

> El objetivo es **entender qué hace el algoritmo**, no memorizar código.


# 1. ¿Qué es KNN?

**K-Nearest Neighbors** o **KNN** es un algoritmo supervisado.

Eso significa que trabajamos con ejemplos para los que ya conocemos:

- sus características;
- la respuesta correcta.

En clasificación, la respuesta es una **clase**.

Cuando llega una observación nueva, KNN pregunta:

> **¿Qué ejemplos conocidos están más cerca?**

Después utiliza las clases de esos vecinos para decidir la predicción.

KNN no aprende una fórmula global explícita. Conserva los ejemplos de entrenamiento y realiza buena parte del trabajo cuando llega una nueva consulta.


# 2. ¿Qué significa “estar cerca”?

Una de las formas más habituales de medir distancia es la **distancia euclídea**.

Para dos puntos:

\[
A=(x_1,y_1)
\]

\[
B=(x_2,y_2)
\]

\[
d(A,B)=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}
\]

Con más variables:

\[
d(\mathbf{x},\mathbf{z})
=
\sqrt{
\sum_{j=1}^{p}(x_j-z_j)^2
}
\]

donde \(p\) es el número de características.


## Ejemplo manual

\[
A=(2,3)
\]

\[
B=(5,7)
\]

\[
d(A,B)=\sqrt{(2-5)^2+(3-7)^2}=5
\]


In [ ]:
import numpy as np

A = np.array([2, 3])
B = np.array([5, 7])

distancia = np.linalg.norm(A - B)

print("Punto A:", A)
print("Punto B:", B)
print("Distancia euclídea:", distancia)


# 3. KNN paso a paso sin `scikit-learn`

Antes de utilizar una librería, veremos una predicción manual.

Crearemos dos grupos de puntos en dos dimensiones.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

X_demo = np.array([
    [1.0, 1.0],
    [1.4, 1.7],
    [2.0, 1.2],
    [2.2, 2.1],
    [4.8, 4.5],
    [5.2, 5.3],
    [5.8, 4.9],
    [6.1, 5.8]
])

y_demo = np.array([0, 0, 0, 0, 1, 1, 1, 1])

df_demo = pd.DataFrame({
    "x1": X_demo[:, 0],
    "x2": X_demo[:, 1],
    "clase": y_demo
})

df_demo


## Visualizar los datos

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df_demo["x1"], df_demo["x2"], c=df_demo["clase"], s=90)
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Datos de entrenamiento")
plt.grid(alpha=0.2)
plt.show()


## Llega una nueva observación

Queremos clasificar:

\[
Q=(3.9,\;4.0)
\]

KNN calculará la distancia entre Q y todos los ejemplos de entrenamiento.


In [ ]:
Q = np.array([3.9, 4.0])

distancias = np.linalg.norm(X_demo - Q, axis=1)

tabla_distancias = df_demo.copy()
tabla_distancias["distancia_a_Q"] = distancias

tabla_distancias.sort_values("distancia_a_Q")


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df_demo["x1"], df_demo["x2"], c=df_demo["clase"], s=90)
plt.scatter(Q[0], Q[1], s=180)
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Nueva observación que queremos clasificar")
plt.grid(alpha=0.2)
plt.show()


# 4. Elegir los `k` vecinos más cercanos

Supongamos:

\[
k=3
\]

Ordenamos las distancias y seleccionamos las tres menores.


In [ ]:
k = 3

indices_ordenados = np.argsort(distancias)
indices_vecinos = indices_ordenados[:k]

vecinos = tabla_distancias.iloc[indices_vecinos].copy()
vecinos


Ahora observamos las clases de esos vecinos y realizamos una votación.

In [ ]:
clases_vecinas = y_demo[indices_vecinos]

valores, conteos = np.unique(clases_vecinas, return_counts=True)
prediccion_manual = valores[np.argmax(conteos)]

print("Clases de los vecinos:", clases_vecinas)
print("Predicción para Q:", prediccion_manual)


## Resumen de una predicción KNN

1. calcular la distancia a los ejemplos de entrenamiento;
2. ordenar las distancias;
3. seleccionar los `k` vecinos más próximos;
4. observar sus clases;
5. realizar una votación;
6. asignar la clase mayoritaria.

Para clasificación:

\[
\hat y =
\operatorname{moda}
(y_{(1)}, y_{(2)}, ..., y_{(k)})
\]


# 5. ¿Qué significa `k`?

`k` indica cuántos vecinos utilizamos.

### `k` pequeño

- decisiones muy locales;
- sensible a observaciones concretas;
- puede sobreajustar.

### `k` grande

- decisiones más suaves;
- puede ser menos sensible al ruido;
- si es demasiado grande puede mezclar estructuras diferentes.

> No existe un `k` universalmente mejor.


In [ ]:
def prediccion_knn_manual(X, y, punto, k):
    distancias = np.linalg.norm(X - punto, axis=1)
    indices = np.argsort(distancias)[:k]
    clases = y[indices]

    valores, conteos = np.unique(clases, return_counts=True)
    prediccion = valores[np.argmax(conteos)]

    return prediccion

for k in [1, 3, 5, 7]:
    pred = prediccion_knn_manual(X_demo, y_demo, Q, k)
    print(f"k={k} -> clase predicha: {pred}")


# 6. El escalado es fundamental

KNN depende directamente de las distancias.

Si una variable tiene valores alrededor de 1–10 y otra alrededor de 10.000–100.000, la segunda puede dominar la distancia simplemente por su escala numérica.

Una estrategia habitual es la estandarización:

\[
z = \frac{x-\mu}{\sigma}
\]

`StandardScaler` expresa las variables en una escala comparable respecto a su media y desviación estándar.

Esto no hace que todas las variables sean igual de importantes conceptualmente; evita que una domine únicamente por sus unidades.


In [ ]:
persona_a = np.array([25, 30000])
persona_b = np.array([45, 30100])

diferencias = np.abs(persona_a - persona_b)

print("Diferencia de edad:", diferencias[0])
print("Diferencia de salario:", diferencias[1])
print("Distancia sin escalar:", np.linalg.norm(persona_a - persona_b))


# 7. KNN real con el dataset Iris

Iris contiene 150 flores de tres especies:

- setosa;
- versicolor;
- virginica.

Cada flor tiene cuatro características:

1. longitud del sépalo;
2. anchura del sépalo;
3. longitud del pétalo;
4. anchura del pétalo.

Nuestro objetivo será predecir la especie.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))

df.head()


## Exploración básica

In [ ]:
print("Dimensiones:", df.shape)

print("\nDistribución por clase:")
print(df["species"].value_counts())

print("\nValores ausentes:")
print(df.isna().sum())

df.describe()


## Gráfico 1 — Pétalos

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    df["petal length (cm)"],
    df["petal width (cm)"],
    c=df["target"],
    s=70
)
plt.xlabel("Longitud del pétalo (cm)")
plt.ylabel("Anchura del pétalo (cm)")
plt.title("Iris: longitud y anchura del pétalo")
plt.grid(alpha=0.2)
plt.show()


Aquí se observa que algunas clases están claramente separadas mientras otras se solapan parcialmente.

Esto ayuda a entender por qué un método basado en vecinos puede funcionar bien.


## Gráfico 2 — Sépalos

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    df["sepal length (cm)"],
    df["sepal width (cm)"],
    c=df["target"],
    s=70
)
plt.xlabel("Longitud del sépalo (cm)")
plt.ylabel("Anchura del sépalo (cm)")
plt.title("Iris: longitud y anchura del sépalo")
plt.grid(alpha=0.2)
plt.show()


Aquí existe más solapamiento.

No todas las variables separan las clases con la misma facilidad.


# 8. Separar entrenamiento y test

- **train**: ejemplos utilizados por el modelo;
- **test**: ejemplos reservados para evaluar.

Usaremos `stratify=y` para conservar aproximadamente la proporción de clases.


In [ ]:
from sklearn.model_selection import train_test_split

X = df[iris.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


# 9. Pipeline: escalado + KNN

Usaremos un `Pipeline` para garantizar que:

1. el escalador se ajusta solo con train;
2. el mismo escalado se aplica a test;
3. después se ejecuta KNN.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

modelo_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

modelo_knn.fit(X_train, y_train)

print("Modelo entrenado.")


## ¿Qué significa `fit()` aquí?

Conceptualmente:

- `StandardScaler` aprende media y desviación de cada característica;
- KNN conserva la información de entrenamiento necesaria para buscar vecinos.

La predicción es donde ocurre buena parte del trabajo.


# 10. Realizar predicciones

In [ ]:
y_pred = modelo_knn.predict(X_test)

resultados = X_test.copy()
resultados["real"] = y_test
resultados["predicho"] = y_pred

resultados["real_nombre"] = resultados["real"].map(
    dict(enumerate(iris.target_names))
)
resultados["predicho_nombre"] = resultados["predicho"].map(
    dict(enumerate(iris.target_names))
)

resultados.head(10)


# 11. Evaluar el modelo

La accuracy representa:

\[
Accuracy =
\frac{\text{predicciones correctas}}
{\text{número total de predicciones}}
\]


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names
    )
)


# 12. Matriz de confusión

- filas: clase real;
- columnas: clase predicha;
- diagonal: predicciones correctas.

Nos permite observar qué clases se confunden entre sí.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=iris.target_names
)

plt.title("Matriz de confusión — KNN")
plt.show()


# 13. Elegir `k` con validación cruzada

No queremos elegir `k` mirando continuamente el conjunto de test.

Utilizaremos validación cruzada sobre train.

Para cada valor de `k`:

1. dividimos temporalmente train en varias partes;
2. entrenamos con unas;
3. validamos con otra;
4. repetimos;
5. calculamos el rendimiento medio.


In [ ]:
from sklearn.model_selection import cross_val_score

valores_k = range(1, 31)

medias_cv = []
desviaciones_cv = []

for k in valores_k:
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    medias_cv.append(scores.mean())
    desviaciones_cv.append(scores.std())

resultados_k = pd.DataFrame({
    "k": list(valores_k),
    "accuracy_cv_media": medias_cv,
    "accuracy_cv_std": desviaciones_cv
})

resultados_k.head()


## Gráfico — rendimiento según `k`

In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(
    resultados_k["k"],
    resultados_k["accuracy_cv_media"]
)
plt.xlabel("Número de vecinos (k)")
plt.ylabel("Accuracy media en validación cruzada")
plt.title("Efecto de k sobre el rendimiento")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
mejor_fila = resultados_k.loc[
    resultados_k["accuracy_cv_media"].idxmax()
]

mejor_k = int(mejor_fila["k"])

print("Mejor k observado:", mejor_k)
print(
    "Accuracy CV media:",
    round(mejor_fila["accuracy_cv_media"], 3)
)


Si varios valores producen resultados muy similares, una diferencia mínima no debe interpretarse como una verdad universal.

La validación cruzada estima el comportamiento del modelo sobre estos datos concretos.


## Modelo final

In [ ]:
modelo_final = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=mejor_k))
])

modelo_final.fit(X_train, y_train)

pred_final = modelo_final.predict(X_test)

accuracy_final = accuracy_score(y_test, pred_final)

print("k final:", mejor_k)
print(f"Accuracy test: {accuracy_final:.3f}")


# 14. Fronteras de decisión

Con cuatro variables no podemos dibujar fácilmente todo el espacio.

Crearemos una versión didáctica con:

- longitud del pétalo;
- anchura del pétalo.

Cada región del gráfico representará la clase que KNN asignaría a una nueva flor situada en esa zona.


In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

X_2d = df[
    ["petal length (cm)", "petal width (cm)"]
]
y_2d = df["target"]

modelo_2d = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

modelo_2d.fit(X_2d, y_2d)

plt.figure(figsize=(9, 6))

DecisionBoundaryDisplay.from_estimator(
    modelo_2d,
    X_2d,
    response_method="predict",
    alpha=0.25
)

plt.scatter(
    X_2d["petal length (cm)"],
    X_2d["petal width (cm)"],
    c=y_2d,
    s=55
)

plt.xlabel("Longitud del pétalo (cm)")
plt.ylabel("Anchura del pétalo (cm)")
plt.title("Fronteras de decisión de KNN con k=5")
plt.show()


La frontera no procede de una recta o ecuación lineal.

Se construye a partir de qué ejemplos quedan más cerca de cada zona.


# 15. Comparar fronteras con distinto `k`

## `k = 1`

In [ ]:
modelo_k1_2d = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=1))
])

modelo_k1_2d.fit(X_2d, y_2d)

plt.figure(figsize=(9, 6))

DecisionBoundaryDisplay.from_estimator(
    modelo_k1_2d,
    X_2d,
    response_method="predict",
    alpha=0.25
)

plt.scatter(
    X_2d["petal length (cm)"],
    X_2d["petal width (cm)"],
    c=y_2d,
    s=55
)

plt.xlabel("Longitud del pétalo (cm)")
plt.ylabel("Anchura del pétalo (cm)")
plt.title("Frontera de decisión — k=1")
plt.show()


## `k = 25`

In [ ]:
modelo_k25_2d = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=25))
])

modelo_k25_2d.fit(X_2d, y_2d)

plt.figure(figsize=(9, 6))

DecisionBoundaryDisplay.from_estimator(
    modelo_k25_2d,
    X_2d,
    response_method="predict",
    alpha=0.25
)

plt.scatter(
    X_2d["petal length (cm)"],
    X_2d["petal width (cm)"],
    c=y_2d,
    s=55
)

plt.xlabel("Longitud del pétalo (cm)")
plt.ylabel("Anchura del pétalo (cm)")
plt.title("Frontera de decisión — k=25")
plt.show()


### Interpretación

Con `k=1` las regiones pueden ser más irregulares y sensibles a observaciones individuales.

Con un `k` mayor suelen suavizarse las decisiones.

Esto conecta con la idea de sobreajuste y generalización.


# 16. Ver los vecinos utilizados en una predicción

Nueva flor:

- longitud del pétalo = 4.8 cm;
- anchura del pétalo = 1.7 cm.


In [ ]:
nueva_flor = pd.DataFrame({
    "petal length (cm)": [4.8],
    "petal width (cm)": [1.7]
})

prediccion = modelo_2d.predict(nueva_flor)[0]

print(
    "Clase predicha:",
    iris.target_names[prediccion]
)


Para conocer los vecinos aplicamos primero el mismo escalado utilizado por el pipeline.


In [ ]:
scaler_2d = modelo_2d.named_steps["scaler"]
knn_2d = modelo_2d.named_steps["knn"]

nueva_flor_escalada = scaler_2d.transform(nueva_flor)

distancias_vecinos, indices_vecinos = knn_2d.kneighbors(
    nueva_flor_escalada,
    n_neighbors=5
)

indices_vecinos = indices_vecinos[0]
distancias_vecinos = distancias_vecinos[0]

tabla_vecinos = X_2d.iloc[indices_vecinos].copy()
tabla_vecinos["clase"] = y_2d.iloc[indices_vecinos].values
tabla_vecinos["especie"] = tabla_vecinos["clase"].map(
    dict(enumerate(iris.target_names))
)
tabla_vecinos["distancia_escalada"] = distancias_vecinos

tabla_vecinos


## Visualizar la flor y sus vecinos

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    X_2d["petal length (cm)"],
    X_2d["petal width (cm)"],
    c=y_2d,
    s=45,
    alpha=0.45
)

vecinos_xy = X_2d.iloc[indices_vecinos]

plt.scatter(
    vecinos_xy["petal length (cm)"],
    vecinos_xy["petal width (cm)"],
    c=y_2d.iloc[indices_vecinos],
    s=140
)

plt.scatter(
    nueva_flor["petal length (cm)"],
    nueva_flor["petal width (cm)"],
    s=220
)

plt.xlabel("Longitud del pétalo (cm)")
plt.ylabel("Anchura del pétalo (cm)")
plt.title("Nueva flor y sus 5 vecinos más próximos")
plt.grid(alpha=0.2)
plt.show()


La predicción puede inspeccionarse directamente: depende de las observaciones situadas alrededor de la nueva muestra.


# 17. ¿Todos los vecinos votan igual?

Por defecto:

```python
weights="uniform"
```

Todos los vecinos tienen el mismo peso.

Otra opción es:

```python
weights="distance"
```

Los vecinos más próximos tienen mayor influencia.

No significa que `distance` sea siempre mejor; es una configuración que puede evaluarse.


In [ ]:
for pesos in ["uniform", "distance"]:
    modelo = Pipeline([
        ("scaler", StandardScaler()),
        (
            "knn",
            KNeighborsClassifier(
                n_neighbors=5,
                weights=pesos
            )
        )
    ])

    scores = cross_val_score(
        modelo,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    print(
        f"{pesos:>8} -> accuracy CV media: {scores.mean():.3f}"
    )


# 18. La métrica de distancia también importa

Además de la distancia euclídea existe, por ejemplo, la distancia Manhattan:

\[
d(\mathbf{x},\mathbf{z})
=
\sum_{j=1}^{p}|x_j-z_j|
\]

La idea importante:

> Si cambiamos la definición de cercanía, pueden cambiar los vecinos y las predicciones.


In [ ]:
for metrica in ["euclidean", "manhattan"]:
    modelo = Pipeline([
        ("scaler", StandardScaler()),
        (
            "knn",
            KNeighborsClassifier(
                n_neighbors=5,
                metric=metrica
            )
        )
    ])

    scores = cross_val_score(
        modelo,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    print(
        f"{metrica:>10} -> accuracy CV media: {scores.mean():.3f}"
    )


# 19. Limitaciones de KNN

## Muchas dimensiones

Cuando el número de características aumenta mucho, las distancias pueden volverse menos informativas. Este problema está relacionado con la **maldición de la dimensionalidad**.

Debemos prestar atención a:

- selección de características;
- escalado;
- dimensionalidad;
- cantidad de datos;
- métrica utilizada.

## Coste de predicción

El entrenamiento puede ser sencillo, pero una predicción puede requerir comparar una nueva observación con muchas observaciones almacenadas.

Con millones de ejemplos, muchas dimensiones y muchas consultas, esto puede ser costoso.


# 20. KNN también puede hacer regresión

En clasificación:

> los vecinos votan una clase.

En regresión:

> combinamos los valores numéricos de los vecinos.

Por ejemplo, con vecinos cuyos valores son:

\[
180,\;200,\;220
\]

una predicción simple sería:

\[
\frac{180+200+220}{3}=200
\]


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

rng = np.random.default_rng(42)

X_reg = np.linspace(0, 10, 45).reshape(-1, 1)

y_reg = (
    np.sin(X_reg[:, 0])
    + rng.normal(0, 0.18, size=len(X_reg))
)

modelo_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5))
])

modelo_reg.fit(X_reg, y_reg)

X_curva = np.linspace(0, 10, 300).reshape(-1, 1)
y_curva = modelo_reg.predict(X_curva)


## Visualizar la regresión KNN

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    X_reg[:, 0],
    y_reg,
    s=55
)

plt.plot(
    X_curva[:, 0],
    y_curva
)

plt.xlabel("x")
plt.ylabel("Variable objetivo")
plt.title("Ejemplo de KNN para regresión")
plt.grid(alpha=0.2)
plt.show()


# 21. Pipeline mental de KNN

1. Definir `X` e `y`.
2. Separar train y test.
3. Escalar las variables.
4. Elegir una configuración inicial.
5. Entrenar con `fit()`.
6. Predecir con `predict()`.
7. Evaluar.
8. Ajustar hiperparámetros con validación.
9. Evaluar una vez el modelo final sobre test.

Hiperparámetros habituales:

- `n_neighbors`;
- `weights`;
- métrica de distancia.


# 22. Errores frecuentes

### No escalar
Las variables con números mayores pueden dominar la distancia.

### Elegir `k` mirando continuamente test
El test deja de ser una evaluación verdaderamente independiente.

### Pensar que `k=1` siempre es mejor
Puede adaptarse demasiado al ruido.

### Pensar que un `k` grande siempre es mejor
Puede suavizar demasiado el modelo.

### Añadir variables irrelevantes sin criterio
También participan en la distancia.

### Pensar que “más cerca” es absoluto
Depende de variables, escala, representación y métrica.


# 23. Mini reto final

Sin copiar directamente las celdas anteriores:

1. carga Iris;
2. utiliza las cuatro características;
3. separa train y test;
4. crea un pipeline con `StandardScaler`;
5. prueba `k = 3`, `5`, `7` y `11`;
6. compara mediante validación cruzada;
7. selecciona una configuración;
8. evalúala sobre test;
9. muestra la matriz de confusión;
10. explica en 5–8 líneas qué significa realmente una predicción de KNN.


In [ ]:
# TU CÓDIGO AQUÍ



# 24. Preguntas de comprensión

1. ¿Por qué KNN es supervisado?
2. ¿Qué representa `k`?
3. ¿Cómo decide una clase?
4. ¿Qué es la distancia euclídea?
5. ¿Por qué el escalado es especialmente importante?
6. ¿Qué podría ocurrir con `k=1`?
7. ¿Qué podría ocurrir con un `k` demasiado grande?
8. ¿Qué diferencia existe entre `uniform` y `distance`?
9. ¿Por qué separamos train y test?
10. ¿Para qué utilizamos validación cruzada al elegir `k`?
11. ¿Qué aporta una matriz de confusión?
12. ¿Qué cambia si utilizamos Manhattan?
13. ¿Qué problema aparece con muchas dimensiones?
14. ¿Por qué la predicción puede ser costosa con millones de ejemplos?
15. ¿Cómo cambia KNN al pasar de clasificación a regresión?


# 25. Qué debes recordar

KNN puede resumirse así:

**nueva observación**

↓

**calcular distancias**

↓

**buscar los k vecinos más próximos**

↓

**observar sus respuestas**

↓

**votar una clase o combinar valores**

Pero para utilizarlo correctamente:

- la escala importa;
- `k` controla cuánto depende el modelo del entorno local;
- la métrica define qué entendemos por cercanía;
- debemos evaluar con datos no utilizados para entrenar;
- KNN puede sobreajustar;
- puede perder eficacia con grandes volúmenes o muchas dimensiones.

> **Idea final:** una predicción puede construirse comparando un nuevo caso con ejemplos conocidos, sin aprender una fórmula global explícita.
